# Validación de `src/optimizacion.py` — Etapa 3

Este notebook confirma que las funciones de `src/optimizacion.py` (frontera eficiente, GMV, tangencia, CML, versiones restringidas) son correctas: se validan contra fórmulas cerradas manuales y contra `PyPortfolioOpt` como referencia independiente. Se reutiliza la misma ventana y los mismos $\mu$ y $\Sigma$ ya validados en la Etapa 2. 

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from pypfopt import EfficientFrontier

from src.config import cargar_config
from src import datos, estimacion, optimizacion

config = cargar_config()
factor_anual = config["estimacion"]["factor_anualizacion"]

## Preparación de datos y estimación de μ, Σ

In [2]:
precios = datos.obtener_precios(config)
benchmark = datos.obtener_benchmark(config)

periodo_inicio = "2025-03-11"
periodo_fin = "2025-09-18"

ventana = precios.loc[periodo_inicio:periodo_fin]
ventana_filtrada = datos.eliminar_columnas_incompletas(ventana, umbral=0.10)
ventana_imputada = datos.imputar_precios(ventana_filtrada)
rendimientos_log = datos.calcular_rendimientos_log(ventana_imputada)

mu = estimacion.estimar_mu_bayes_stein(rendimientos_log, factor_anual=factor_anual)
Sigma = estimacion.estimar_sigma_ledoit_wolf(rendimientos_log, factor_anual=factor_anual)

r_f = 0.02

print(f"Shape rendimientos: {rendimientos_log.shape}")
print("mu: Bayes-Stein, Sigma: Ledoit-Wolf")
print(f"Tasa libre de riesgo de validacion: {r_f}")

Shape rendimientos: (132, 110)
mu: Bayes-Stein, Sigma: Ledoit-Wolf
Tasa libre de riesgo de validacion: 0.02


## Portafolio de mínima varianza (GMV) — sin restricciones

`pesos_gmv` usa la fórmula cerrada $\displaystyle \alpha_{mv}= \frac{\Sigma^{-1}\,\mathbf{1}}{\mathbf{1}^T\,\Sigma^{-1}\,\mathbf{1}}$. Verificamos que coincide con `PyPortfolioOpt` permitiendo ventas en corto y que su punto $(\sigma, \mu)$ cae exactamente sobre `frontera_eficiente_sin_restricciones`.

In [3]:
pesos_gmv = optimizacion.pesos_gmv(Sigma)

ef_gmv = EfficientFrontier(mu, Sigma, weight_bounds=(None, None))
ef_gmv.min_volatility()
pesos_gmv_pypfopt = np.array(list(ef_gmv.clean_weights().values()))

diff_gmv = np.max(np.abs(pesos_gmv.values - pesos_gmv_pypfopt))
print(f"Diferencia maxima pesos_gmv vs PyPortfolioOpt: {diff_gmv:.2e}")
assert diff_gmv < 1e-3

resultado_frontera = optimizacion.frontera_eficiente_sin_restricciones(mu, Sigma, n_points=300)

mu_gmv_manual = float(mu.values @ pesos_gmv.values)
sigma_gmv_manual = float(np.sqrt(pesos_gmv.values @ Sigma.values @ pesos_gmv.values))

print(f"(r_gmv, sigma_gmv) via formula cerrada de la frontera: ({resultado_frontera['r_gmv']:.6f}, {resultado_frontera['sigma_gmv']:.6f})")
print(f"(mu, sigma) del portafolio pesos_gmv: ({mu_gmv_manual:.6f}, {sigma_gmv_manual:.6f})")

assert np.isclose(resultado_frontera["r_gmv"], mu_gmv_manual, atol=1e-6)
assert np.isclose(resultado_frontera["sigma_gmv"], sigma_gmv_manual, atol=1e-4)
print("pesos_gmv coincide con PyPortfolioOpt y cae sobre la frontera cerrada")

Diferencia maxima pesos_gmv vs PyPortfolioOpt: 5.01e-06
(r_gmv, sigma_gmv) via formula cerrada de la frontera: (0.046206, 0.030486)
(mu, sigma) del portafolio pesos_gmv: (0.046206, 0.030486)
pesos_gmv coincide con PyPortfolioOpt y cae sobre la frontera cerrada


## Frontera eficiente sin restricciones — validación cruzada

Barremos varios retornos objetivo con `pesos_frontera` y confirmamos que la varianza resultante coincide con la fórmula de la curva completa `frontera_eficiente_sin_restricciones`.

In [4]:
r_prueba = np.linspace(resultado_frontera["r_gmv"], mu.max() * 0.5, 10)
sigma_manual = []

for r in r_prueba:
    pesos_r = optimizacion.pesos_frontera(mu, Sigma, r)
    assert np.isclose(pesos_r.sum(), 1.0, atol=1e-8)
    sigma_manual.append(np.sqrt(pesos_r.values @ Sigma.values @ pesos_r.values))

sigma_manual = np.array(sigma_manual)
sigma_formula = np.sqrt(np.interp(r_prueba, resultado_frontera["r_completa"], resultado_frontera["sigma_completa"] ** 2))

diff_curva = np.max(np.abs(sigma_manual - sigma_formula))
print(f"Diferencia maxima sigma (pesos_frontera vs curva cerrada): {diff_curva:.2e}")
assert diff_curva < 1e-3
print("pesos_frontera coincide con frontera_eficiente_sin_restricciones para todo el rango de retornos")

Diferencia maxima sigma (pesos_frontera vs curva cerrada): 3.16e-06
pesos_frontera coincide con frontera_eficiente_sin_restricciones para todo el rango de retornos


## Portafolio de tangencia y verificación de tangencia real con la CML

Primero mostramos un hallazgo esperado: la fórmula cerrada del portafolio de tangencia es singular cuando $r_f$ se acerca al retorno del GMV ($r_{mv} \approx 0.046$ en esta ventana) — el denominador $\mathbf{1}^T\,\Sigma^{-1}(\mu − r_f\,\mathbf{1})$ tiende a cero y los pesos explotan. Esto es una propiedad matemática conocida de la frontera sin restricciones, no un error de implementación. Elegimos $r_f = 0.02$ (alejado de $r_{mv}$) para las validaciones.

In [5]:
for r_f_prueba in [0.03, resultado_frontera["r_gmv"], 0.05]:
    pesos_r_f = optimizacion.pesos_tangencia(mu, Sigma, r_f_prueba)
    print(f"r_f={r_f_prueba:.4f} -> peso maximo absoluto: {pesos_r_f.abs().max():.2f}")

r_f=0.0300 -> peso maximo absoluto: 0.60
r_f=0.0462 -> peso maximo absoluto: 485474897845626.62
r_f=0.0500 -> peso maximo absoluto: 2.47


In [6]:
pesos_tan = optimizacion.pesos_tangencia(mu, Sigma, r_f)

ef_tan = EfficientFrontier(mu, Sigma, weight_bounds=(None, None))
ef_tan.max_sharpe(risk_free_rate=r_f)
pesos_tan_pypfopt = np.array(list(ef_tan.clean_weights().values()))

diff_tan = np.max(np.abs(pesos_tan.values - pesos_tan_pypfopt))
print(f"Diferencia maxima pesos_tangencia vs PyPortfolioOpt: {diff_tan:.2e}")
assert diff_tan < 1e-3

mu_tan = float(mu.values @ pesos_tan.values)
sigma_tan = float(np.sqrt(pesos_tan.values @ Sigma.values @ pesos_tan.values))
pendiente_cml = (mu_tan - r_f) / sigma_tan

eps = 1e-5
pesos_izq = optimizacion.pesos_frontera(mu, Sigma, mu_tan - eps)
pesos_der = optimizacion.pesos_frontera(mu, Sigma, mu_tan + eps)
sigma_izq = np.sqrt(pesos_izq.values @ Sigma.values @ pesos_izq.values)
sigma_der = np.sqrt(pesos_der.values @ Sigma.values @ pesos_der.values)
pendiente_frontera = (2 * eps) / (sigma_der - sigma_izq)

print(f"Pendiente CML: {pendiente_cml:.4f}")
print(f"Pendiente de la frontera en tangencia: {pendiente_frontera:.4f}")
assert np.isclose(pendiente_cml, pendiente_frontera, rtol=1e-3)
print("La CML es tangente a la frontera en el portafolio de tangencia")

Diferencia maxima pesos_tangencia vs PyPortfolioOpt: 9.15e-06
Pendiente CML: 8.9557
Pendiente de la frontera en tangencia: 8.9557
La CML es tangente a la frontera en el portafolio de tangencia


## Versión restringida (long-only): GMV y tangencia

Con `permitir_cortos=False`, los pesos deben sumar 1, estar en $[0, 1]$, y la varianza del GMV restringido debe ser mayor o igual a la del GMV sin restricciones (el conjunto factible se reduce).

In [7]:
pesos_gmv_r = optimizacion.pesos_gmv_restringido(Sigma)

ef_gmv_r = EfficientFrontier(mu, Sigma, weight_bounds=(0, 1))
ef_gmv_r.min_volatility()
pesos_gmv_r_pypfopt = np.array(list(ef_gmv_r.clean_weights().values()))

diff_gmv_r = np.max(np.abs(pesos_gmv_r.values - pesos_gmv_r_pypfopt))
print(f"Diferencia maxima pesos_gmv_restringido vs PyPortfolioOpt: {diff_gmv_r:.2e}")
assert diff_gmv_r < 1e-2

assert np.isclose(pesos_gmv_r.sum(), 1.0, atol=1e-6)
assert pesos_gmv_r.min() >= -1e-8
assert pesos_gmv_r.max() <= 1 + 1e-8

var_gmv_sin = pesos_gmv.values @ Sigma.values @ pesos_gmv.values
var_gmv_con = pesos_gmv_r.values @ Sigma.values @ pesos_gmv_r.values
print(f"Varianza GMV sin restricciones: {var_gmv_sin:.6f}")
print(f"Varianza GMV restringido: {var_gmv_con:.6f}")
assert var_gmv_con >= var_gmv_sin - 1e-8

Diferencia maxima pesos_gmv_restringido vs PyPortfolioOpt: 8.48e-04
Varianza GMV sin restricciones: 0.000929
Varianza GMV restringido: 0.000953


In [8]:
pesos_tan_r = optimizacion.pesos_tangencia_restringida(mu, Sigma, r_f)

ef_tan_r = EfficientFrontier(mu, Sigma, weight_bounds=(0, 1))
ef_tan_r.max_sharpe(risk_free_rate=r_f)
pesos_tan_r_pypfopt = np.array(list(ef_tan_r.clean_weights().values()))

diff_tan_r = np.max(np.abs(pesos_tan_r.values - pesos_tan_r_pypfopt))
print(f"Diferencia maxima pesos_tangencia_restringida vs PyPortfolioOpt: {diff_tan_r:.2e}")
assert diff_tan_r < 1e-2

assert np.isclose(pesos_tan_r.sum(), 1.0, atol=1e-6)
assert pesos_tan_r.min() >= -1e-8
assert pesos_tan_r.max() <= 1 + 1e-8

Diferencia maxima pesos_tangencia_restringida vs PyPortfolioOpt: 3.19e-05


## Frontera eficiente restringida

Barrido numérico (`SLSQP`) entre el GMV restringido y el retorno máximo alcanzable. Verificamos que la curva es monótona creciente en $\sigma$ y que su punto inicial coincide con el GMV restringido.

In [9]:
resultado_restringido = optimizacion.frontera_eficiente_restringida(mu, Sigma, n_points=15)

print(f"r_gmv restringido: {resultado_restringido['r_gmv']:.6f}")
print(f"sigma_gmv restringido: {resultado_restringido['sigma_gmv']:.6f}")

assert np.all(np.diff(resultado_restringido["sigma_eficiente"]) >= -1e-8)
assert np.isclose(resultado_restringido["sigma_eficiente"][0], resultado_restringido["sigma_gmv"], atol=1e-6)

r_gmv restringido: 0.049928
sigma_gmv restringido: 0.030873


## Resumen uniforme para la interfaz: `resumen_portafolio` y `optimizar_portafolio`

In [10]:
resumen_gmv = optimizacion.optimizar_portafolio(mu, Sigma, tipo="gmv", permitir_cortos=True)
resumen_tan = optimizacion.optimizar_portafolio(mu, Sigma, tipo="tangencia", r_f=r_f, permitir_cortos=True)
resumen_obj = optimizacion.optimizar_portafolio(mu, Sigma, tipo="objetivo", r_objetivo=0.10, permitir_cortos=False)

for nombre, resumen in [("GMV", resumen_gmv), ("Tangencia", resumen_tan), ("Objetivo (restringido)", resumen_obj)]:
    print(f"--- {nombre} ---")
    print(f"mu: {resumen['mu']:.4f}, sigma: {resumen['sigma']:.4f}")
    if "sharpe" in resumen:
        print(f"sharpe: {resumen['sharpe']:.4f}")
    print(f"suma pesos_pct: {resumen['pesos_pct'].sum():.4f}")
    display(resumen["pesos_pct"].head())
    print()
    assert np.isclose(resumen["pesos_pct"].sum(), 100.0, atol=1e-3)

--- GMV ---
mu: 0.0462, sigma: 0.0305
sharpe: 1.5156
suma pesos_pct: 100.0000


POCHTECB.MX    2.298625
ACTINVRB.MX    2.209687
FINAMEXO.MX    2.119116
VINTE.MX       2.087874
ALTERNA.MX     2.039361
dtype: float64


--- Tangencia ---
mu: 2.8645, sigma: 0.3176
sharpe: 8.9557
suma pesos_pct: 100.0000


CIEB.MX        37.331085
GBMO.MX        22.676206
MEDICAB.MX     22.655699
CEMEXCPO.MX    22.178093
GENTERA.MX     18.351497
dtype: float64


--- Objetivo (restringido) ---
mu: 0.1000, sigma: 0.0316
sharpe: 3.1653
suma pesos_pct: 100.0000


ACTINVRB.MX    2.509777
POCHTECB.MX    2.419513
CIEB.MX        2.411646
FINAMEXO.MX    2.149919
ALTERNA.MX     2.104737
dtype: float64